# AI Receipt & Expense Automation Assistant

Step 1: Setting up your environment, securely entering your Gemini API key, and building the core AI Receipt Parser function.

What We Are Doing in Step 1
Installing Packages: We install the modern google-genai SDK and pydantic (which ensures our output strictly follows a clean JSON structure).

Secure Key Entry: We use Colab's built-in userdata feature (or getpass) to safely handle your API key.

Structured Prompting: We write a Python function that sends raw receipt text to the gemini-2.5-flash model and forces it to return clean, structured JSON matching an expense schema.

In [4]:
# Change this:
# model = "gemini-2.0-flash"

# To this:
model = "gemini-3.6-flash"

In [6]:
# ==========================================
# STEP 1: Dependencies & Core Receipt Parser (FIXED MODEL)
# ==========================================

# 1. Install modern Google GenAI SDK and Pydantic
!pip install -q google-genai pydantic

import json
from getpass import getpass
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# 2. Securely set up Gemini API Key
import os
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")

# Initialize client
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# 3. Define the desired JSON schema using Pydantic
class ReceiptData(BaseModel):
    merchant: str = Field(description="Name of the store or merchant")
    date: str = Field(description="Date of purchase in YYYY-MM-DD format")
    total_amount: float = Field(description="Total transaction amount")
    category: str = Field(description="Category e.g., Dining, Groceries, Utilities, Travel, Retail")
    tax_amount: float = Field(description="Tax amount paid, 0.0 if not listed")

# 4. Core Parser Function using gemini-3.6-flash
def parse_receipt_text(raw_text: str) -> dict:
    """Translates unstructured receipt text into structured JSON using Gemini API."""
    prompt = f"""
    You are an expert financial AI support parser. Extract key expense data from the following receipt text.

    Receipt Text:
    {raw_text}
    """

    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ReceiptData,
            temperature=0.1
        )
    )

    return json.loads(response.text)

# ==========================================
# TEST STEP 1
# ==========================================
sample_receipt = """
Starbucks Coffee #1042
Date: 2026-09-15
1x Iced Latte - $6.50
1x Blueberry Muffin - $4.00
Subtotal: $10.50
Tax: $0.84
TOTAL: $11.34
Thank you for visiting!
"""

print("Processing receipt...")
extracted_json = parse_receipt_text(sample_receipt)
print("\nExtracted Structured JSON:")
print(json.dumps(extracted_json, indent=2))

Processing receipt...

Extracted Structured JSON:
{
  "merchant": "Starbucks Coffee #1042",
  "date": "2026-09-15",
  "total_amount": 11.34,
  "category": "Dining",
  "tax_amount": 0.84
}


In [9]:
# ==========================================
# STEP 2: Database Setup (SQLite + ChromaDB) - FIXED
# ==========================================

# 1. Install ChromaDB for Vector Search
!pip install -q chromadb

import sqlite3
import chromadb
from chromadb.utils import embedding_functions

# 2. Setup SQLite Database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS expenses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    merchant TEXT,
    date TEXT,
    total_amount REAL,
    category TEXT,
    tax_amount REAL
)
""")
conn.commit()

# 3. Setup ChromaDB Vector Store (get_or_create to prevent duplicate errors)
chroma_client = chromadb.Client()
default_ef = embedding_functions.DefaultEmbeddingFunction()
vector_collection = chroma_client.get_or_create_collection(
    name="expense_vectors",
    embedding_function=default_ef
)

# 4. Helper Function to Store Expense in SQL + Vector DB
def store_expense(parsed_json: dict):
    cursor.execute("""
        INSERT INTO expenses (merchant, date, total_amount, category, tax_amount)
        VALUES (?, ?, ?, ?, ?)
    """, (
        parsed_json["merchant"],
        parsed_json["date"],
        parsed_json["total_amount"],
        parsed_json["category"],
        parsed_json["tax_amount"]
    ))
    conn.commit()
    sql_id = cursor.lastrowid

    summary_text = f"Spent ${parsed_json['total_amount']} at {parsed_json['merchant']} for {parsed_json['category']} on {parsed_json['date']}."

    vector_collection.add(
        documents=[summary_text],
        metadatas=[{"sql_id": sql_id, "category": parsed_json["category"]}],
        ids=[f"doc_{sql_id}_{time.time()}"]  # Unique ID using timestamp
    )

    print(f"Stored expense ID #{sql_id}: {summary_text}")

# ==========================================
# TEST STEP 2: Add Batch Receipts
# ==========================================
import time

sample_receipts = [
    """Starbucks Coffee - Date: 2026-09-15 - Total: $11.34 - Category: Dining""",
    """Uber Ride NYC - Date: 2026-09-14 - Total: $34.50 - Category: Travel""",
    """Walmart Grocery - Date: 2026-09-13 - Total: $85.20 - Category: Groceries""",
    """Shell Gas Station - Date: 2026-09-12 - Total: $45.00 - Category: Travel"""
]

print("Processing batch receipts...\n")
for r_text in sample_receipts:
    parsed = parse_receipt_text(r_text)
    store_expense(parsed)

print("\nAll receipts processed and stored in SQLite + Vector DB successfully!")

Processing batch receipts...



/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 28.1MiB/s]


Stored expense ID #1: Spent $11.34 at Starbucks Coffee for Dining on 2026-09-15.
Stored expense ID #2: Spent $34.5 at Uber Ride NYC for Travel on 2026-09-14.
Stored expense ID #3: Spent $85.2 at Walmart Grocery for Groceries on 2026-09-13.
Stored expense ID #4: Spent $45.0 at Shell Gas Station for Travel on 2026-09-12.

All receipts processed and stored in SQLite + Vector DB successfully!


In [ ]:
# ==========================================
# STEP 3: RAG Search Engine & Gradio Web Frontend
# ==========================================

# 1. Install Gradio for UI
!pip install -q gradio

import gradio as gr

# 2. RAG Query Function using gemini-3.6-flash
def answer_expense_question(user_question: str) -> str:
    """Uses ChromaDB vector search + Gemini to answer spending questions."""
    # Retrieve top 2 matching document summaries from ChromaDB
    results = vector_collection.query(
        query_texts=[user_question],
        n_results=2
    )

    retrieved_docs = results.get("documents", [[]])[0]
    context_text = "\n".join(retrieved_docs) if retrieved_docs else "No relevant expenses found."

    # Prompt Gemini using the retrieved vector context
    rag_prompt = f"""
    You are an AI Expense Support Assistant. Answer the user's question using ONLY the provided receipt context.

    Retrieved Receipt Context:
    {context_text}

    User Question: {user_question}

    Provide a concise, direct answer summarizing the spending.
    """

    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=rag_prompt
    )

    return response.text

# 3. Gradio Interface Event Handlers
def process_new_receipt_ui(raw_text):
    if not raw_text.strip():
        return "Please paste receipt text."
    parsed = parse_receipt_text(raw_text)
    store_expense(parsed)
    return f"Success! Added Expense:\n{json.dumps(parsed, indent=2)}"

def ask_question_ui(question):
    if not question.strip():
        return "Please ask a question."
    return answer_expense_question(question)

# 4. Build the Gradio App
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧾 AI Receipt & Expense Automation Assistant")
    gr.Markdown("Built for Recipto AI Support Engineer Demo | **Stack:** Gemini API (`gemini-3.6-flash`), SQLite, ChromaDB, RAG")

    with gr.Row():
        # Left Column: Parse & Store
        with gr.Column():
            gr.Markdown("### 1. Extract & Store Receipt")
            receipt_input = gr.Textbox(
                lines=5,
                placeholder="Paste raw receipt text here...",
                label="Raw Receipt Text"
            )
            parse_btn = gr.Button("Process Receipt", variant="primary")
            parse_output = gr.Code(label="Extracted JSON & Database Status", language="json")

        # Right Column: RAG Search Engine
        with gr.Column():
            gr.Markdown("### 2. Query Spending (RAG Engine)")
            question_input = gr.Textbox(
                lines=2,
                placeholder="e.g., How much did I spend on Uber or travel?",
                label="Ask Question"
            )
            query_btn = gr.Button("Search Expenses", variant="secondary")
            query_output = gr.Textbox(lines=4, label="AI Answer (Powered by Vector Search)")

    # Define Event Handlers
    parse_btn.click(fn=process_new_receipt_ui, inputs=receipt_input, outputs=parse_output)
    query_btn.click(fn=ask_question_ui, inputs=question_input, outputs=query_output)

# Launch UI directly inside Colab with a temporary public link
demo.launch(share=True, debug=True)

/tmp/ipykernel_634/2109681400.py:55: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ec066d3cdeedc1c089.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
